# EE542 Lab 6 — CUDA matrix multiplication

Runtime → Change runtime type → **T4 GPU**, then run the cells in order.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

Get the code.

In [ ]:
REPO = 'https://github.com/Tr3nt-Xie/ee542-lab6.git'
!git clone -q $REPO lab6 || true
%cd lab6

In [ ]:
!make clean && make all   # ARCH=native picks whatever GPU Colab assigned (T4 = sm_75)

## Part 1 — CPU baselines
Same arithmetic, different loop order: watch the GFLOP/s column.

In [ ]:
!for N in 256 512 1024; do bin/matmul_cpu naive $N 1; bin/matmul_cpu ikj $N 1; done
!OMP_NUM_THREADS=$(nproc) bin/matmul_cpu_omp omp 1024 1

## Parts 2–6 — every GPU implementation at one size
Columns: impl, N, kernel_ms, GFLOP/s, end-to-end ms, max relative error vs cuBLAS.

In [ ]:
!bin/sgemm all 2048 10

## Full sweep and plot

In [ ]:
!bash scripts/run_all.sh
!python3 scripts/plot.py results/results.csv

In [ ]:
from IPython.display import Image
Image('results/results.png')

## Part 7 — the CUDA shared library from Python
`handout` allocates and copies on every call; `ctx_*` keeps device buffers alive. Compare call time with kernel time.

In [ ]:
!python3 python/bench_matmul.py 256 512 1024 2048 4096

## Part 8 — convolution: filters, edge detection, performance
Put your own photos in `conv/images/` first if you want them used.

In [ ]:
!python3 python/conv_demo.py

In [ ]:
from IPython.display import Image, display
import glob
for g in sorted(glob.glob('results/conv/gallery_*.png')):
    display(Image(g))

## Optional — profile one kernel with Nsight Compute
Shows achieved memory bandwidth, compute throughput and why warps stall.

In [ ]:
!ncu --set basic -k regex:k_tiled -c 1 bin/sgemm tiled16 2048 1 2>&1 | tail -40